In [10]:
import numpy as np
import pandas as pd

from sklearn.cross_decomposition import PLSRegression
from sklearn.metrics import mean_squared_error

file_path = "../data/10_dropped_variables_and_renamed.csv"
df = pd.read_csv(file_path)

df["week"] = pd.to_datetime(df["week"], dayfirst=True)
df = df.sort_values(["coin", "week"]).reset_index(drop=True)

print(df.head())
print(df["coin"].value_counts().sort_index())

  coin       week    ret_1w  volume_coin_lag1  volume_usd_lag1  rsi_14_lag1  \
0  BCH 2018-01-07  0.066400        1381032.70     3.618961e+09    78.941312   
1  BCH 2018-01-14 -0.002797        1652361.39     4.074998e+09    79.654435   
2  BCH 2018-01-21 -0.294976        1180978.74     2.979197e+09    79.524175   
3  BCH 2018-01-28 -0.014861        1307532.01     2.496130e+09    67.095855   
4  BCH 2018-02-04 -0.324328         961985.35     1.581405e+09    66.697136   

   ma_4_lag1   ma_12_lag1  volatility_4w_lag1  volatility_12w_lag1  ...  \
0  2077.8025  1318.362500            0.359122             0.410451  ...   
1  2386.5425  1503.091667            0.311423             0.402309  ...   
2  2556.9950  1686.270000            0.288544             0.404304  ...   
3  2302.4750  1796.930833            0.160971             0.429015  ...   
4  2147.3325  1890.903333            0.159677             0.427459  ...   

   gt_ETC_lag1  gt_DASH_lag1  gt_IOTA_lag1  gt_OMG_lag1  gt_XMR_lag1  \
0 

Create one dataframe per coin

In [11]:
df_btc = df[df["coin"] == "BTC"].copy()
df_eth = df[df["coin"] == "ETH"].copy()
df_xrp = df[df["coin"] == "XRP"].copy()
df_ltc = df[df["coin"] == "LTC"].copy()
df_bch = df[df["coin"] == "BCH"].copy()
df_eos = df[df["coin"] == "EOS"].copy()
df_etc = df[df["coin"] == "ETC"].copy()
df_dash = df[df["coin"] == "DASH"].copy()
df_iota = df[df["coin"] == "IOTA"].copy()
df_omg = df[df["coin"] == "OMG"].copy()
df_xmr = df[df["coin"] == "XMR"].copy()
df_zec = df[df["coin"] == "ZEC"].copy()

Function to tune PLS for one coin 

In [18]:
def tune_pls_h1_coin(
    df_coin,
    max_comp=10,
    train_end="2022-12-30",
    valid_start="2023-01-01",
    valid_end="2024-12-30",
    target_col="ret_1w",
    date_col="week",
    coin_col="coin",
    exclude_cols=None,
    verbose=True
):
    if exclude_cols is None:
        exclude_cols = [
            coin_col,
            date_col,
            "ret_1w",
            "log_ret_1w",
            "ret_1w_original",
            "log_ret_1w_original"
        ]

    df_coin = df_coin.copy()
    df_coin[date_col] = pd.to_datetime(df_coin[date_col], dayfirst=True)
    df_coin = df_coin.sort_values(date_col).reset_index(drop=True)

    train_end = pd.Timestamp(train_end)
    valid_start = pd.Timestamp(valid_start)
    valid_end = pd.Timestamp(valid_end)

    predictor_cols = [c for c in df_coin.columns if c not in exclude_cols]

    train_mask = df_coin[date_col] <= train_end
    valid_mask = (df_coin[date_col] >= valid_start) & (df_coin[date_col] <= valid_end)

    if train_mask.sum() == 0:
        raise ValueError("no training rows found")

    if valid_mask.sum() == 0:
        raise ValueError("no validation rows found")

    valid_indices = df_coin.index[valid_mask].tolist()
    n_features = len(predictor_cols)
    max_comp = min(max_comp, n_features)

    rmse_by_comp = {}
    detail_rows = []

    coin_name = df_coin[coin_col].iloc[0]

    for k in range(1, max_comp + 1):
        preds = []
        actuals = []

        if verbose:
            print(f"{coin_name}: tuning pls with {k} component(s)...")

        for idx in valid_indices:
            train_df = df_coin.iloc[:idx].copy()
            test_df = df_coin.iloc[[idx]].copy()

            x_train = train_df[predictor_cols].copy()
            y_train = train_df[target_col].copy()

            x_test = test_df[predictor_cols].copy()
            y_test = test_df[target_col].copy()

            x_medians = x_train.median()
            x_train = x_train.fillna(x_medians)
            x_test = x_test.fillna(x_medians)

            x_means = x_train.mean()
            x_sds = x_train.std(ddof=0).replace(0, 1.0)

            x_train_scaled = (x_train - x_means) / x_sds
            x_test_scaled = (x_test - x_means) / x_sds

            model = PLSRegression(n_components=k)
            model.fit(x_train_scaled, y_train)

            y_pred = model.predict(x_test_scaled).ravel()[0]

            preds.append(float(y_pred))
            actuals.append(float(y_test.iloc[0]))

            detail_rows.append({
                "coin": coin_name,
                "ncomp": k,
                "week": test_df[date_col].iloc[0],
                "actual": float(y_test.iloc[0]),
                "pred": float(y_pred)
            })

        rmse_k = np.sqrt(mean_squared_error(actuals, preds))
        rmse_by_comp[k] = rmse_k

        if verbose:
            print(f"{coin_name}: validation rmse = {rmse_k:.6f}")

    best_ncomp = min(rmse_by_comp, key=rmse_by_comp.get)
    rmse_detail = pd.DataFrame(detail_rows)

    return {
        "coin": coin_name,
        "best_ncomp": best_ncomp,
        "rmse_by_comp": rmse_by_comp,
        "rmse_detail": rmse_detail,
        "predictor_cols": predictor_cols
    }

Function to fit final model for one coin on train+validation

In [19]:
def build_pls_h1_coin(
    df_coin,
    h1_comp,
    train_valid_end="2024-12-30",
    target_col="ret_1w",
    date_col="week",
    coin_col="coin",
    exclude_cols=None
):
    if exclude_cols is None:
        exclude_cols = [
            coin_col,
            date_col,
            "ret_1w",
            "log_ret_1w",
            "ret_1w_original",
            "log_ret_1w_original"
        ]

    df_coin = df_coin.copy()
    df_coin[date_col] = pd.to_datetime(df_coin[date_col], dayfirst=True)
    df_coin = df_coin.sort_values(date_col).reset_index(drop=True)

    train_valid_end = pd.Timestamp(train_valid_end)
    train_df = df_coin[df_coin[date_col] <= train_valid_end].copy()

    predictor_cols = [c for c in train_df.columns if c not in exclude_cols]

    x_train = train_df[predictor_cols].copy()
    y_train = train_df[target_col].copy()

    x_medians = x_train.median()
    x_train = x_train.fillna(x_medians)

    x_means = x_train.mean()
    x_sds = x_train.std(ddof=0).replace(0, 1.0)

    x_train_scaled = (x_train - x_means) / x_sds

    model = PLSRegression(n_components=h1_comp)
    model.fit(x_train_scaled, y_train)

    return {
        "coin": train_df[coin_col].iloc[0],
        "h1_comp": h1_comp,
        "predictor_cols": predictor_cols,
        "x_medians": x_medians,
        "x_means": x_means,
        "x_sds": x_sds,
        "fitted_model": model
    }

Predict test set for one coin and create outputs

In [20]:
def predict_pls_h1_coin(
    df_coin,
    pls_struct,
    test_start="2025-01-01",
    test_end="2026-01-08",
    target_col="ret_1w",
    date_col="week",
    coin_col="coin"
):
    df_coin = df_coin.copy()
    df_coin[date_col] = pd.to_datetime(df_coin[date_col], dayfirst=True)
    df_coin = df_coin.sort_values(date_col).reset_index(drop=True)

    test_start = pd.Timestamp(test_start)
    test_end = pd.Timestamp(test_end)

    test_df = df_coin[
        (df_coin[date_col] >= test_start) &
        (df_coin[date_col] <= test_end)
    ].copy()

    predictor_cols = pls_struct["predictor_cols"]
    x_medians = pls_struct["x_medians"]
    x_means = pls_struct["x_means"]
    x_sds = pls_struct["x_sds"]
    model = pls_struct["fitted_model"]

    x_test = test_df[predictor_cols].copy()
    x_test = x_test.fillna(x_medians)
    x_test_scaled = (x_test - x_means) / x_sds

    pred = model.predict(x_test_scaled).ravel()

    pred_df = test_df[[coin_col, date_col]].copy()
    pred_df["actual"] = test_df[target_col].values
    pred_df["pred"] = pred

    resid_df = pred_df.copy()
    resid_df["residual"] = resid_df["actual"] - resid_df["pred"]

    rmse = np.sqrt(mean_squared_error(pred_df["actual"], pred_df["pred"]))

    return pred_df, resid_df, rmse

Run for 12 coin-specific execution chunks.

BTC: 

In [21]:
res_btc = tune_pls_h1_coin(df_btc, max_comp=10, verbose=False)
btc_comp = res_btc["best_ncomp"]
btc_model = build_pls_h1_coin(df_btc, h1_comp=btc_comp)
btc_pred, btc_resid, btc_rmse = predict_pls_h1_coin(df_btc, btc_model)

btc_pred.to_excel("../outputs/btc_pls_predictions.xlsx", index=False)
btc_resid.to_excel("../outputs/btc_pls_residuals.xlsx", index=False)

print("btc optimal components:", btc_comp)
print("btc rmse:", btc_rmse)

btc optimal components: 1
btc rmse: 0.053575731565945785


ETH:

In [22]:
res_eth = tune_pls_h1_coin(df_eth, max_comp=10, verbose=False)
eth_comp = res_eth["best_ncomp"]
eth_model = build_pls_h1_coin(df_eth, h1_comp=eth_comp)
eth_pred, eth_resid, eth_rmse = predict_pls_h1_coin(df_eth, eth_model)

eth_pred.to_excel("../outputs/eth_pls_predictions.xlsx", index=False)
eth_resid.to_excel("../outputs/eth_pls_residuals.xlsx", index=False)

print("eth optimal components:", eth_comp)
print("eth rmse:", eth_rmse)

eth optimal components: 2
eth rmse: 0.11453662622080935


XRP:

In [23]:
res_xrp = tune_pls_h1_coin(df_xrp, max_comp=10, verbose=False)
xrp_comp = res_xrp["best_ncomp"]
xrp_model = build_pls_h1_coin(df_xrp, h1_comp=xrp_comp)
xrp_pred, xrp_resid, xrp_rmse = predict_pls_h1_coin(df_xrp, xrp_model)

xrp_pred.to_excel("../outputs/xrp_pls_predictions.xlsx", index=False)
xrp_resid.to_excel("../outputs/xrp_pls_residuals.xlsx", index=False)

print("xrp optimal components:", xrp_comp)
print("xrp rmse:", xrp_rmse)

xrp optimal components: 1
xrp rmse: 0.12114089769898467


In [24]:
res_ltc = tune_pls_h1_coin(df_ltc, max_comp=10, verbose=False)
ltc_comp = res_ltc["best_ncomp"]
ltc_model = build_pls_h1_coin(df_ltc, h1_comp=ltc_comp)
ltc_pred, ltc_resid, ltc_rmse = predict_pls_h1_coin(df_ltc, ltc_model)

ltc_pred.to_excel("../outputs/ltc_pls_predictions.xlsx", index=False)
ltc_resid.to_excel("../outputs/ltc_pls_residuals.xlsx", index=False)

print("ltc optimal components:", ltc_comp)
print("ltc rmse:", ltc_rmse)

ltc optimal components: 3
ltc rmse: 0.11164153190891087


In [25]:
res_bch = tune_pls_h1_coin(df_bch, max_comp=10, verbose=False)
bch_comp = res_bch["best_ncomp"]
bch_model = build_pls_h1_coin(df_bch, h1_comp=bch_comp)
bch_pred, bch_resid, bch_rmse = predict_pls_h1_coin(df_bch, bch_model)

bch_pred.to_excel("../outputs/bch_pls_predictions.xlsx", index=False)
bch_resid.to_excel("../outputs/bch_pls_residuals.xlsx", index=False)

print("bch optimal components:", bch_comp)
print("bch rmse:", bch_rmse)

bch optimal components: 1
bch rmse: 0.08397859057651974


In [26]:
res_eos = tune_pls_h1_coin(df_eos, max_comp=10, verbose=False)
eos_comp = res_eos["best_ncomp"]
eos_model = build_pls_h1_coin(df_eos, h1_comp=eos_comp)
eos_pred, eos_resid, eos_rmse = predict_pls_h1_coin(df_eos, eos_model)

eos_pred.to_excel("../outputs/eos_pls_predictions.xlsx", index=False)
eos_resid.to_excel("../outputs/eos_pls_residuals.xlsx", index=False)

print("eos optimal components:", eos_comp)
print("eos rmse:", eos_rmse)

eos optimal components: 2
eos rmse: 0.5541982521945784


In [27]:
res_etc = tune_pls_h1_coin(df_etc, max_comp=10, verbose=False)
etc_comp = res_etc["best_ncomp"]
etc_model = build_pls_h1_coin(df_etc, h1_comp=etc_comp)
etc_pred, etc_resid, etc_rmse = predict_pls_h1_coin(df_etc, etc_model)

etc_pred.to_excel("../outputs/etc_pls_predictions.xlsx", index=False)
etc_resid.to_excel("../outputs/etc_pls_residuals.xlsx", index=False)

print("etc optimal components:", etc_comp)
print("etc rmse:", etc_rmse)

etc optimal components: 2
etc rmse: 0.12043186510372243


In [28]:
res_dash = tune_pls_h1_coin(df_dash, max_comp=10, verbose=False)
dash_comp = res_dash["best_ncomp"]
dash_model = build_pls_h1_coin(df_dash, h1_comp=dash_comp)
dash_pred, dash_resid, dash_rmse = predict_pls_h1_coin(df_dash, dash_model)

dash_pred.to_excel("../outputs/dash_pls_predictions.xlsx", index=False)
dash_resid.to_excel("../outputs/dash_pls_residuals.xlsx", index=False)

print("dash optimal components:", dash_comp)
print("dash rmse:", dash_rmse)

dash optimal components: 2
dash rmse: 0.15982995978003975


In [29]:
res_iota = tune_pls_h1_coin(df_iota, max_comp=10, verbose=False)
iota_comp = res_iota["best_ncomp"]
iota_model = build_pls_h1_coin(df_iota, h1_comp=iota_comp)
iota_pred, iota_resid, iota_rmse = predict_pls_h1_coin(df_iota, iota_model)

iota_pred.to_excel("../outputs/iota_pls_predictions.xlsx", index=False)
iota_resid.to_excel("../outputs/iota_pls_residuals.xlsx", index=False)

print("iota optimal components:", iota_comp)
print("iota rmse:", iota_rmse)

iota optimal components: 1
iota rmse: 0.15184170999468252


In [30]:
res_omg = tune_pls_h1_coin(df_omg, max_comp=10, verbose=False)
omg_comp = res_omg["best_ncomp"]
omg_model = build_pls_h1_coin(df_omg, h1_comp=omg_comp)
omg_pred, omg_resid, omg_rmse = predict_pls_h1_coin(df_omg, omg_model)

omg_pred.to_excel("../outputs/omg_pls_predictions.xlsx", index=False)
omg_resid.to_excel("../outputs/omg_pls_residuals.xlsx", index=False)

print("omg optimal components:", omg_comp)
print("omg rmse:", omg_rmse)

omg optimal components: 2
omg rmse: 0.12244947498810375


In [31]:
res_xmr = tune_pls_h1_coin(df_xmr, max_comp=10, verbose=False)
xmr_comp = res_xmr["best_ncomp"]
xmr_model = build_pls_h1_coin(df_xmr, h1_comp=xmr_comp)
xmr_pred, xmr_resid, xmr_rmse = predict_pls_h1_coin(df_xmr, xmr_model)

xmr_pred.to_excel("../outputs/xmr_pls_predictions.xlsx", index=False)
xmr_resid.to_excel("../outputs/xmr_pls_residuals.xlsx", index=False)

print("xmr optimal components:", xmr_comp)
print("xmr rmse:", xmr_rmse)

xmr optimal components: 2
xmr rmse: 0.15207512131637285


In [32]:
res_zec = tune_pls_h1_coin(df_zec, max_comp=10, verbose=False)
zec_comp = res_zec["best_ncomp"]
zec_model = build_pls_h1_coin(df_zec, h1_comp=zec_comp)
zec_pred, zec_resid, zec_rmse = predict_pls_h1_coin(df_zec, zec_model)

zec_pred.to_excel("../outputs/zec_pls_predictions.xlsx", index=False)
zec_resid.to_excel("../outputs/zec_pls_residuals.xlsx", index=False)

print("zec optimal components:", zec_comp)
print("zec rmse:", zec_rmse)

zec optimal components: 2
zec rmse: 0.27020947306557197


Single summary table: 

In [33]:
summary_df = pd.DataFrame({
    "coin": ["btc", "eth", "xrp", "ltc", "bch", "eos", "etc", "dash", "iota", "omg", "xmr", "zec"],
    "best_ncomp": [btc_comp, eth_comp, xrp_comp, ltc_comp, bch_comp, eos_comp, etc_comp, dash_comp, iota_comp, omg_comp, xmr_comp, zec_comp],
    "rmse": [btc_rmse, eth_rmse, xrp_rmse, ltc_rmse, bch_rmse, eos_rmse, etc_rmse, dash_rmse, iota_rmse, omg_rmse, xmr_rmse, zec_rmse]
})

summary_df.to_excel("../outputs/pls_coin_summary.xlsx", index=False)
print(summary_df)

    coin  best_ncomp      rmse
0    btc           1  0.053576
1    eth           2  0.114537
2    xrp           1  0.121141
3    ltc           3  0.111642
4    bch           1  0.083979
5    eos           2  0.554198
6    etc           2  0.120432
7   dash           2  0.159830
8   iota           1  0.151842
9    omg           2  0.122449
10   xmr           2  0.152075
11   zec           2  0.270209
